# Inputs

In [2]:
import sys
sys.path.append('../')
import transport_frames.src.graph_builder.criteria as criteria
import transport_frames.src.metrics.indicators as indicators
from transport_frames.src.metrics.indicators import visualize_availability
import transport_frames.src.metrics.grade_territory as grade_territory
import transport_frames.src.graph_builder.graphbuilder as graphbuilder
import momepy
import osmnx as ox
import geopandas as gpd
import shapely
import pandas as pd
import networkx as nx
import numpy as np
import json

/Users/sashamorozov/Documents/Code/NCCR/transport_frames/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def create_json_file(d1, filename):
    json_dict = {}
    for key in d1.keys():
        if isinstance(d1[key],(float,int)):
            json_dict[key] = d1[key]
        else:
            json_dict[key] = d1[key].to_json()
    json_object = json.dumps(json_dict)
    with open(filename, 'w') as f:
        f.write(json_object)

In [17]:
# Ленинградская_область EPSG: 32636  / 176095 / N27025179 (Гатчина)
# Санкт_Петербург EPSG:32636 / 337422 / N27490597
# Москва EPSG:32637 / 102269 / N1686293227
# Волгоградская_область EPSG:32638 / 77665 /  N27504363
# Тульская_область EPSG:32637 /  81993 / N34389350
# Омская_область EPSG:32643 / 140292 /  N27503946
# Краснодарский_край EPSG:32637 / 108082 / N27505129
# Тюменская_область EPSG:32642 / 140291 / N27505666
# Московская_область EPSG:32637 / 51490 / N1686293227

In [3]:
region_name = 'Ленинградская_область'
local_crs = 32636
geocode = 176095
region_capital = ox.geocode_to_gdf('N27025179',by_osmid=True)

In [4]:
MO_polygon = ox.geocode_to_gdf('R176095', by_osmid=True).to_crs(epsg=32636)
MSK_polygon = ox.geocode_to_gdf('R337422', by_osmid=True).to_crs(epsg=32636).buffer(3000)
city = gpd.GeoDataFrame(
    MO_polygon.union(MSK_polygon).to_crs(epsg=4326),
    geometry=MO_polygon.union(MSK_polygon).to_crs(epsg=4326),
    crs="EPSG:4326"  # Установка CRS (например, WGS84)
)

In [4]:
city = ox.geocode_to_gdf(f'R{geocode}', by_osmid=True).to_crs(epsg=4326)
city['name'] = region_name

In [5]:
city['name'] = region_name
city.explore()

In [6]:
# lo_polygon = ox.geocode_to_gdf('R176095', by_osmid=True).to_crs(epsg=32636)
# spb_polygon = ox.geocode_to_gdf('R337422', by_osmid=True).to_crs(epsg=32636).buffer(3000)
# city = lo_polygon.union(spb_polygon).to_crs(epsg=4326) #  get lo polygon
# city['geometry'] = lo_polygon.union(spb_polygon).to_crs(epsg=4326)
# city['name'] = region_name

russia = ox.geocode_to_gdf("Russia") #  get border of the country

regions = gpd.read_file('/Users/sashamorozov/Downloads/regions_of_russia.geojson') #  get regions
regions = regions[regions['ISO3166-2']!='RU-CHU']
regions = regions.to_crs(city.crs)

In [7]:
polygon = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Тульская_область/Тульская_область_region.geojson')

In [43]:
# points = polygons188.copy()
# # points['geometry'] = shapely.centroid(polygons188['geometry']).set_crs(polygons188.crs)
# points['geometry'] = points['geometry'].apply(lambda geom: geom.representative_point())

In [21]:

PLACEHOLDER = gpd.GeoDataFrame(geometry=[]) # set where are no services 

polygons188 = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Ленинградская_область/Ленинградская_область_settlement.geojson')
polygons18 = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Ленинградская_область/Ленинградская_область_district.geojson')

# points = shapely.centroid(polygons188.geometry).set_crs(polygons188.crs)

# points = polygons188.copy()
# points['geometry'] = shapely.centroid(polygons188['geometry']).set_crs(polygons188.crs)
# points['geometry'] = points['geometry'].apply(lambda geom: geom.representative_point())

points = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Ленинградская_область/Ленинградская_область_region_points.geojson')
points188 = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Ленинградская_область/Ленинградская_область_settlement_admin_centres_188_nodes.geojson')
points18 = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Ленинградская_область/Ленинградская_область_district_admin_centres_18_nodes.geojson')

# points188 = shapely.centroid(polygons188.geometry).set_crs(polygons188.crs)
# points18 = points[points['is_admin_centre_district']==True]


stops = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Ленинградская_область/Ленинградская_область_railway_stops.geojson')

fuel = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Ленинградская_область/Ленинградская_область_fuel_stations.geojson')

ferry = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Ленинградская_область/Ленинградская_область_ferry_terminal.geojson')

aero = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Ленинградская_область/Ленинградская_область_aerodrome_local.geojson')

international_aero = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Ленинградская_область/Ленинградская_область_international_aerodrome.geojson')
# international_aero = aero[aero['type']=='international']
international_aero['geometry'] = shapely.centroid(international_aero['geometry']).set_crs(international_aero.crs)

# aero = aero[aero['type']!='international']
aero['geometry'] = shapely.centroid(aero['geometry']).set_crs(aero.crs)


water_objects = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Ленинградская_область/Ленинградская_область_water.geojson')
oopt = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Ленинградская_область/Ленинградская_область_nature_reserve.geojson')

train_paths = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Ленинградская_область/Ленинградская_область_railway_roads.geojson')

bus_routes = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Ленинградская_область/Ленинградская_область_bus_routes.geojson')
bus_stops = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Ленинградская_область/Ленинградская_область_bus_stops.geojson')
# russia = ox.geocode_to_gdf("Russia") 
# regions = gpd.read_file(f'/Users/sashamorozov/Downloads/regions_of_russia.geojson') #  get regions
# regions = regions[regions['ISO3166-2']!='RU-CHU']
# regions = regions.to_crs(city.crs)

# test territory
# neud = polygons188.iloc[[0]]
# p1 = gpd.read_file('/Users/sashamorozov/Downloads/lo_gdfs/Аэродром лодейнопольское поселение.geojson')
# p2 = gpd.read_file('/Users/sashamorozov/Downloads/lo_gdfs/Аэродром Сиверск .geojson')
# p3 = gpd.read_file('/Users/sashamorozov/Downloads/lo_gdfs/project Светогорского поселения.geojson')
# p4 = gpd.read_file('/Users/sashamorozov/Downloads/lo_gdfs/project Шлиссельбург.geojson')
# neud = pd.concat([p1,p2,p3,p4])

In [11]:
neud = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Ленинградская_область/Аэродром Сиверск.geojson')

In [ ]:
citygraph, neud,
                                    regions_gdf=polygons18,
                                    districts_gdf=polygons188,
                                    region_capital=region_capital,
                                    region_centers=points18,
                                    district_centers=polygons188,
                                    settlement_centers=points,
                                    fuel=fuel,
                                    train_stops=stops,
                                    international_aero=international_aero,
                                    aero=aero,
                                    ports=ferry,
                                    water_objects=water_objects, 
                                    oopt=oopt, 
                                    train_paths=train_paths, 
                                    inter = None,
                                    bus_stops=bus_stops, 
                                    bus_routes=bus_routes,crs=local_crs

# Preprocessing

In [19]:
import os

directory = f"../preprocessed/{region_name}/"
os.makedirs(directory, exist_ok=True)

In [34]:
# write an uds graph
citygraph = graphbuilder.get_graph_from_polygon(city, crs=local_crs,country_polygon=russia)
citygraph_ = graphbuilder.convert_list_attr_to_str(citygraph)
nx.write_graphml(citygraph_, f"../preprocessed/{region_name}/{region_name}_uds.graphml")

/Users/sashamorozov/Documents/Code/NCCR/transport_frames/.venv/lib/python3.10/site-packages/osmnx/_overpass.py:254: UserWarning: This area is 25 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


In [35]:
citygraph = indicators.prepare_graph(citygraph)

In [ ]:
n,e = momepy.nx_to_gdf(citygraph)
e.explore()

In [14]:
points18

,x,y,border,name188,name18,geometry
0,33.845997,59.471209,0.0,Бокситогорск,Бокситогорск,POINT (33.84600 59.47121)
1,28.600008,59.378977,0.0,Кингисепп,Кингисепп,POINT (28.60001 59.37898)
2,32.009647,59.450580,0.0,Кириши,Кириши,POINT (32.00965 59.45058)
3,30.981783,59.875390,0.0,Кировск,Кировск,POINT (30.98178 59.87539)
4,33.541450,59.644131,0.0,Тихвин,Тихвин,POINT (33.54145 59.64413)
5,33.543355,60.734316,0.0,Лодейное Поле,Лодейное Поле,POINT (33.54335 60.73432)
6,30.122304,59.569061,0.0,Гатчина,Гатчина,POINT (30.12230 59.56906)
7,34.164883,60.912602,0.0,Подпорожье,Подпорожье,POINT (34.16488 60.91260)
8,30.112834,61.050364,0.0,Приозерск,Приозерск,POINT (30.11283 61.05036)
9,29.091986,59.904514,0.0,Сосновый бор,Сосновый бор,POINT (29.09199 59.90451)


In [14]:
# # write inter into graphml
inter = indicators.get_intermodal(geocode,local_crs)
e = [
        (u, v, k)
        for u, v, k, d in inter.edges(data=True, keys=True)
        if d.get("type") in (['bus','walk'])
    ]
inter = inter.edge_subgraph(e)

# save routes and stops
ni,ei = momepy.nx_to_gdf(inter)
bus_routes = ei[ei['type']=='bus']
bus_routes.to_file(f'../preprocessed/{region_name}/{region_name}_bus_routes.geojson')

# convert to file
inter_= graphbuilder.convert_list_attr_to_str(inter)
for node,e,data in inter_.edges(data=True):
    if 'geometry' in data:
        data['geometry']=str(data['geometry'])
nx.write_graphml(inter_, f"../preprocessed/{region_name}/{region_name}_inter.graphml")

2024-06-16 17:22:40.552 | INFO     | dongraphio.dongraphio:get_intermodal_graph_from_osm:55 - Creating intermodal graph from OSM...
2024-06-16 17:22:40.554 | WARNING  | dongraphio.base_models.grapher_logic:get_public_trasport_graph:236 - Files with routes or with stops was not found. The graph will be built based on data from OSM
2024-06-16 17:34:25.672 | INFO     | dongraphio.base_models.grapher_logic:get_public_trasport_graph:284 - Public transport graph done!
2024-06-16 17:34:26.318 | DEBUG    | dongraphio.base_models.grapher_logic:get_osmnx_graph:94 - Extracting and preparing walk graph from OSM ...
/Users/sashamorozov/Documents/Code/NCCR/transport_frames/.venv/lib/python3.10/site-packages/osmnx/_overpass.py:254: UserWarning: This area is 24 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)
2024-06-16 17:54:

In [ ]:
inter = indicators.prepare_graph(inter)

In [53]:
# citygraph = nx.read_graphml('/Users/sashamorozov/Documents/Code/NCCR/transport_frames/preprocessed/Санкт-Петербург/Санкт-Петербург_uds.graphml')
# citygraph = graphbuilder.convert_list_attr_from_str(citygraph)
# citygraph = indicators.prepare_graph(citygraph)

In [10]:
inter = nx.read_graphml('/Users/sashamorozov/Documents/Code/NCCR/transport_frames/preprocessed/Санкт-Петербург/Санкт-Петербург_inter.graphml')
inter = graphbuilder.convert_list_attr_from_str(inter)
inter = indicators.prepare_graph(inter)


In [9]:
# save routes and stops
ni,ei = momepy.nx_to_gdf(inter)
bus_routes = ei[ei['type']=='bus']
bus_routes.to_file(f'../preprocessed/{region_name}/{region_name}_bus_routes.geojson')

/var/folders/zx/byb3bcds50s1lwbdr2mwq18r0000gn/T/ipykernel_10353/3479026531.py:2: UserWarning: Approach is not set. Defaulting to 'primal'.
  ni,ei = momepy.nx_to_gdf(inter)


In [15]:

# save routes and stops
ni,ei = momepy.nx_to_gdf(inter)
bus_routes = ei[ei['type']=='bus']
bus_routes.to_file(f'../preprocessed/{region_name}/{region_name}_bus_routes.geojson')

# convert to file
inter_= graphbuilder.convert_list_attr_to_str(inter)
for node,e,data in inter_.edges(data=True):
    if 'geometry' in data:
        data['geometry']=str(data['geometry'])
nx.write_graphml(inter_, f"../preprocessed/{region_name}/{region_name}_inter.graphml")

/var/folders/zx/byb3bcds50s1lwbdr2mwq18r0000gn/T/ipykernel_10140/540198691.py:10: UserWarning: Approach is not set. Defaulting to 'primal'.
  ni,ei = momepy.nx_to_gdf(inter)
Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x1051ba650>>
Traceback (most recent call last):
  File "/Users/sashamorozov/Documents/Code/NCCR/transport_frames/.venv/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


In [10]:
frame = nx.read_graphml('/Users/sashamorozov/Documents/Code/NCCR/transport_frames/preprocessed/Санкт-Петербург/Санкт-Петербург_frame.graphml')
frame = graphbuilder.convert_list_attr_from_str(frame)
frame = indicators.prepare_graph(frame)

In [23]:
inter_= graphbuilder.convert_list_attr_to_str(inter)
for node,e,data in inter_.edges(data=True):
    if 'geometry' in data:
        data['geometry']=str(data['geometry'])
nx.write_graphml(inter_, f"../preprocessed/{region_name}/{region_name}_inter.graphml")

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x105cba650>>
Traceback (most recent call last):
  File "/Users/sashamorozov/Documents/Code/NCCR/transport_frames/.venv/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


In [39]:
# write frame into graphml
frame = graphbuilder.get_frame(citygraph,regions.to_crs(city.crs),city)
frame = graphbuilder.assign_city_names_to_nodes(points18.reset_index().to_crs(frame.graph['crs']), 
                                                momepy.nx_to_gdf(frame)[0], frame, name_col='name18', max_distance=1200)
frame_ = frame.copy()
for node,e,data in frame_.edges(data=True):
    if 'geometry' in data:
        data['geometry']=str(data['geometry'])
for node,data in frame_.nodes(data=True):
    if 'ref' in data:
        data['ref']=str(data['ref'])
frame_ = graphbuilder.convert_list_attr_to_str(frame_)
nx.write_graphml(frame_, f"../preprocessed/{region_name}/{region_name}_frame.graphml")

/var/folders/zx/byb3bcds50s1lwbdr2mwq18r0000gn/T/ipykernel_7831/2731190791.py:2: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  frame = graphbuilder.get_frame(citygraph,regions.to_crs(city.crs),city)
/Users/sashamorozov/Documents/Code/NCCR/transport_frames/transport_frames/src/graph_builder/graphbuilder.py:505: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.08]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  n.loc[(n["nodeID"] == path[j]), "weight"] += weight
/Users/sashamorozov/Documents/Code/NCCR/transport_frames/transport_frames/src/graph_builder/graphbuilder.py:506: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.08]' has dtype incompatible with int64, please explicitly cast to a compat

In [40]:
frame = indicators.prepare_graph(frame)

In [6]:
inter = nx.read_graphml('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/graphs/Тюменская_область/Тюменская_область_inter.graphml')
inter = graphbuilder.convert_list_attr_from_str(inter)
inter = indicators.prepare_graph(inter)

ni,ei = momepy.nx_to_gdf(inter)
bus_routes = ei[ei['type']=='bus']
bus_routes.to_file(f'../preprocessed/{region_name}/{region_name}_bus_routes.geojson')

/var/folders/zx/byb3bcds50s1lwbdr2mwq18r0000gn/T/ipykernel_1120/2861731210.py:5: UserWarning: Approach is not set. Defaulting to 'primal'.
  ni,ei = momepy.nx_to_gdf(inter)


# Processing

In [24]:
citygraph = nx.read_graphml(f'../preprocessed/{region_name}/{region_name}_uds.graphml')
citygraph = graphbuilder.convert_list_attr_from_str(citygraph)
citygraph = indicators.prepare_graph(citygraph)

In [9]:
citygraph = nx.read_graphml('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/graphs/Ленинградская_область/Ленинградская_область_uds.graphml')
citygraph = graphbuilder.convert_list_attr_from_str(citygraph)
citygraph = indicators.prepare_graph(citygraph)

In [66]:
n,e = momepy.nx_to_gdf(frame)


In [7]:
inter = nx.read_graphml(f"../preprocessed/{region_name}/{region_name}_inter.graphml")
inter = graphbuilder.convert_list_attr_from_str(inter)
inter = indicators.prepare_graph(inter)

In [20]:
inter = nx.read_graphml('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/graphs/Ленинградская_область/Ленинградская_область_inter.graphml')
inter = graphbuilder.convert_list_attr_from_str(inter)
inter = indicators.prepare_graph(inter)

ni,ei = momepy.nx_to_gdf(inter)
bus_routes = ei[ei['type']=='bus']
bus_routes.to_file(f'../preprocessed/{region_name}/{region_name}_bus_routes.geojson')

/var/folders/zx/byb3bcds50s1lwbdr2mwq18r0000gn/T/ipykernel_2523/2531259429.py:5: UserWarning: Approach is not set. Defaulting to 'primal'.
  ni,ei = momepy.nx_to_gdf(inter)


In [12]:
bus_routes = gpd.read_file('/Users/sashamorozov/Documents/Code/NCCR/transport_frames/preprocessed/Санкт-Петербург/Санкт-Петербург_bus_routes.geojson')

# # or if we don't have them:
# ni,ei = momepy.nx_to_gdf(inter)
# bus_routes = ei[ei['type']=='bus']

In [8]:
frame = nx.read_graphml(f"../preprocessed/{region_name}/{region_name}_frame.graphml")
frame = graphbuilder.convert_list_attr_from_str(frame)
frame = indicators.prepare_graph(frame)

In [13]:
frame = nx.read_graphml('/Users/sashamorozov/Documents/Code/NCCR/transport_frames/preprocessed/Санкт-Петербург/Санкт-Петербург_frame.graphml')
frame = graphbuilder.convert_list_attr_from_str(frame)
frame = indicators.prepare_graph(frame)

# region

In [17]:
d0 = indicators.indicator_area(citygraph, city, points, inter,region_capital, 
                               fuel, stops, international_aero, aero, ferry,train_paths,bus_stops, crs=local_crs)
for k,v in d0.items():
    if isinstance(v,gpd.GeoDataFrame):
        cols_to_drop = [col for col in v.columns if col not in ['index', 'name', 'layer', 'density', 'geometry', 'reg1_length', 'reg2_length', 'reg3_length', 'to_service', 'index', 'number_of_routes', 'service_count', 'total_length_km']]
        v.drop(cols_to_drop, axis=1, inplace=True)
create_json_file(d0,f"../preprocessed/{region_name}/{region_name}_indicators_region.json")

2024-06-16 18:41:52.270 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:84 - Creating adjacency matrix based on provided graph...
2024-06-16 18:42:20.690 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances ...
2024-06-16 18:55:18.139 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:93 - Adjacency matrix done!
/Users/sashamorozov/Documents/Code/NCCR/transport_frames/.venv/lib/python3.10/site-packages/numpy/core/_methods.py:118: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
2024-06-16 18:55:19.113 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:84 - Creating adjacency matrix based on provided graph...
2024-06-16 18:57:14.982 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances ...
2024-06-16 19:12:58.979 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:93 - Adjacency matrix done!
2024-06-16 19:13:00.651 | INF

Some services cannot be reached from some nodes of the graph. The nodes were removed from analysis


2024-06-16 19:25:23.545 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances ...
2024-06-16 19:25:31.356 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:93 - Adjacency matrix done!
2024-06-16 19:25:31.389 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:84 - Creating adjacency matrix based on provided graph...


Some services cannot be reached from some nodes of the graph. The nodes were removed from analysis


2024-06-16 19:25:41.065 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances ...
2024-06-16 19:28:06.555 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:93 - Adjacency matrix done!
2024-06-16 19:28:06.629 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:84 - Creating adjacency matrix based on provided graph...


...

2024-06-16 19:30:52.252 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances ...
2024-06-16 19:31:22.106 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:93 - Adjacency matrix done!
2024-06-16 19:31:22.136 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:84 - Creating adjacency matrix based on provided graph...


Some services cannot be reached from some nodes of the graph. The nodes were removed from analysis


2024-06-16 19:31:56.758 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances ...
2024-06-16 19:32:22.450 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:93 - Adjacency matrix done!


Some services cannot be reached from some nodes of the graph. The nodes were removed from analysis


/Users/sashamorozov/Documents/Code/NCCR/transport_frames/transport_frames/src/metrics/indicators.py:402: UserWarning: Approach is not set. Defaulting to 'primal'.
  ei = momepy.nx_to_gdf(inter)[1]


In [35]:
def read_json_file(filename):
    with open(filename, 'r') as f:
        json_object = json.load(f)

    result_dict = {}

    for key in json_object.keys():
        if isinstance(json_object[key], str):  # check if the value is a string
            try:
                gdf = gpd.read_file(json_object[key])
                result_dict[key] = gdf
            except:
                try:
                    result_dict[key] = json.loads(json_object[key])
                except json.JSONDecodeError:
                    result_dict[key] = json_object[key]
        else:
            result_dict[key] = json_object[key]

    return result_dict

d0 = read_json_file('/Users/sashamorozov/Documents/Code/NCCR/transport_frames/preprocessed/Московская область/Московская область_indicators_region.json')

print(d0.keys()) # check all possible keys
# a['connectivity'] # check what's stored inside

dict_keys(['aggregated_density', 'road_length_gdf', 'connectivity', 'connectivity_public_transport', 'to_fed_roads', 'to_region_admin_center', 'azs_availability', 'number_of_fuel_stations', 'train_stops_availability', 'number_of_train_stops', 'international_aero_availability', 'number_of_international_aero', 'local_aero_availability', 'number_of_local_aero', 'port_availability', 'number_of_ports', 'number_of_bus_stops', 'train_paths_length', 'number_of_bus_routes'])


In [36]:
d0.keys()

dict_keys(['aggregated_density', 'road_length_gdf', 'connectivity', 'connectivity_public_transport', 'to_fed_roads', 'to_region_admin_center', 'azs_availability', 'number_of_fuel_stations', 'train_stops_availability', 'number_of_train_stops', 'international_aero_availability', 'number_of_international_aero', 'local_aero_availability', 'number_of_local_aero', 'port_availability', 'number_of_ports', 'number_of_bus_stops', 'train_paths_length', 'number_of_bus_routes'])

In [38]:
d0['connectivity']

,id,index,name,to_service,geometry
0,0,0,Московская область,1.70141,"POLYGON ((36.88530 55.22933, 36.88347 55.23001..."


In [39]:
ind_dict = {

    'to_fed_roads': {
        'column': 'to_service',
        '№ п/п':'3.1.1.2',
        'Название хранимое':'Средняя удаленность до федеральных транспортных магистралей',
        'ед.изм.': 'км'

    },

    'to_region_admin_center': {
        'column': 'to_service',
        '№ п/п':'3.1.1.4',
        'Название хранимое':'Средняя удаленность до центра региона',
        'ед.изм.': 'км'

    },


    'aggregated_density': {
        'column': 'density',
        '№ п/п':'3.1.3',
        'Название хранимое':'Плотность улично-дорожной сети',
        'ед.изм.': 'км/км2'

    },

    'road_length_gdf': {
        'reg1_length': {
            'column': 'reg1_length',
            '№ п/п':'3.1.4.1',
            'Название хранимое':'Протяженность дорог федерального значения',
            'ед.изм.': 'км'
        },
        'reg2_length': {
            'column': 'reg2_length',
            '№ п/п':'3.1.4.2',
            'Название хранимое':'Протяженность дорог регионального значения',
            'ед.изм.': 'км'
        },
        'reg3_length': {
            'column': 'reg3_length',
            '№ п/п':'3.1.4.3',
            'Название хранимое':'Протяженность дорог местного значения',
            'ед.изм.': 'км'
        }
    }, 

    'number_of_bus_routes': {
        'column': 'number_of_routes',
        '№ п/п':'3.1.5.1.1',
        'Название хранимое':'Количество маршрутов общественного транспорта',
        'ед.изм.': 'ед'

    },

    'number_of_bus_stops': {
        'column': 'service_count',
        '№ п/п':'3.1.5.1.2',
        'Название хранимое':'Количество остановок общественного транспорта',
        'ед.изм.': 'ед'

    },
    
    'number_of_fuel_stations': {
        'column': 'service_count',
        '№ п/п':'3.1.5.2.1',
        'Название хранимое':'Количество автозаправочных станций',
        'ед.изм.': 'ед'

    },

    'azs_availability': {
        'column': 'to_service',
        '№ п/п':'3.1.5.2.2',
        'Название хранимое':'Средняя доступность автозаправочных станций',
        'ед.изм.': 'мин'

    },

     'train_paths_length': {
        'column': 'total_length_km',
        '№ п/п':'3.2.1',
        'Название хранимое':'Общая протяженность железнодорожных путей',
        'ед.изм.': 'км'

    },

    'number_of_train_stops': {
        'column': 'service_count',
        '№ п/п':'3.2.2',
        'Название хранимое':'Количество остановок железнодорожного транспорта',
        'ед.изм.': 'ед'

    },

    'train_stops_availability': {
        'column': 'to_service',
        '№ п/п':'3.2.3',
        'Название хранимое':'Средняя доступность остановок железнодорожного транспорта',
        'ед.изм.': 'мин'

    },

    'number_of_international_aero': {
        'column': 'service_count',
        '№ п/п':'3.3.1.1',
        'Название хранимое':'Количество международных аэропортов',
        'ед.изм.': 'ед'

    },

    'number_of_local_aero': {
        'column': 'service_count',
        '№ п/п':'3.3.1.2',
        'Название хранимое':'Количество аэропортов местного значения',
        'ед.изм.': 'ед'

    },

    'international_aero_availability': {
        'column': 'to_service',
        '№ п/п':'3.3.2.1',
        'Название хранимое':'Средняя доступность международных аэропортов',
        'ед.изм.': 'мин'

    },

    'local_aero_availability': {
        'column': 'to_service',
        '№ п/п':'3.3.2.2',
        'Название хранимое':'Средняя доступность аэропортов местного значения',
        'ед.изм.': 'мин'

    },

    'number_of_ports': {
        'column': 'service_count',
        '№ п/п':'3.4.1',
        'Название хранимое':'Количество портов',
        'ед.изм.': 'ед'

   }

}

gdfs = []
for gdf_name, gdf_dict in ind_dict.items():
    tmp_gdf = d0[gdf_name].copy()
    if 'column' in gdf_dict:
        tmp_gdf = tmp_gdf[['name', gdf_dict['column']]].rename(columns={
            'name': 'Территория',
            gdf_dict['column']: 'Значение'
        })
        for key, value in gdf_dict.items():
            if key == 'column':
                continue
            tmp_gdf[key] = value
        gdfs.append(tmp_gdf)
    else:
        for sub_key, sub_dict in gdf_dict.items():
            tmp_gdf_sub = tmp_gdf[['name', sub_dict['column']]].rename(columns={
                'name': 'Территория',
                sub_dict['column']: 'Значение'
            })
            for key, value in sub_dict.items():
                if key == 'column':
                    continue
                tmp_gdf_sub[key] = value
            gdfs.append(tmp_gdf_sub)

dfdf = pd.concat(gdfs)
dfdf = dfdf[['№ п/п','Название хранимое','ед.изм.','Значение','Территория']]

dfdf = dfdf.assign(
    Источник='modeled',
    Период='2024'
)
dfdf.to_excel('Московская_область_район.xlsx', index=False)
dfdf


,№ п/п,Название хранимое,ед.изм.,Значение,Территория,Источник,Период
0,3.1.1.2,Средняя удаленность до федеральных транспортны...,км,9.919423,Московская область,modeled,2024
0,3.1.1.4,Средняя удаленность до центра региона,км,97.182432,Московская область,modeled,2024
0,3.1.3,Плотность улично-дорожной сети,км/км2,2645.678000,Московская область,modeled,2024
0,3.1.4.1,Протяженность дорог федерального значения,км,5720.435614,Московская область,modeled,2024
0,3.1.4.2,Протяженность дорог регионального значения,км,33730.126723,Московская область,modeled,2024
0,3.1.4.3,Протяженность дорог местного значения,км,84965.635578,Московская область,modeled,2024
0,3.1.5.1.1,Количество маршрутов общественного транспорта,ед,1918.000000,Московская область,modeled,2024
0,3.1.5.1.2,Количество остановок общественного транспорта,ед,19017.000000,Московская область,modeled,2024
0,3.1.5.2.1,Количество автозаправочных станций,ед,1004.000000,Московская область,modeled,2024
0,3.1.5.2.2,Средняя доступность автозаправочных станций,мин,6.681033,Московская область,modeled,2024


# District

In [40]:
d1 = indicators.indicator_area(citygraph, polygons18, points, inter,region_capital, fuel, stops, international_aero, aero, ferry,train_paths,bus_stops, crs=local_crs)

for k,v in d1.items():
    if isinstance(v,gpd.GeoDataFrame):
        cols_to_drop = [col for col in v.columns if col not in ['index', 'name', 'layer', 'density', 'geometry', 'reg1_length', 'reg2_length', 'reg3_length', 'to_service', 'index', 'number_of_routes', 'service_count', 'total_length_km']]
        v.drop(cols_to_drop, axis=1, inplace=True)
create_json_file(d1,f"../preprocessed/{region_name}/{region_name}_indicators_mr.json")

/Users/sashamorozov/Documents/Code/NCCR/transport_frames/transport_frames/src/metrics/indicators.py:293: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  result = polygons_gdf.merge(length_sums, how='left', left_on='index', right_on='index').fillna(0)
2024-06-16 19:56:14.232 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:84 - Creating adjacency matrix based on provided graph...
2024-06-16 19:56:24.819 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances ...
2024-06-16 20:09:40.395 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:93 - Adjacency matrix done!
/Users/sashamorozov/Documents/Code/NCCR/transport_frames/.venv/lib/python3.10/site-packages/numpy/core/_methods.py:118: RuntimeWarning: overflow encountered in 

Some services cannot be reached from some nodes of the graph. The nodes were removed from analysis


2024-06-16 20:40:41.481 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances ...
2024-06-16 20:40:49.648 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:93 - Adjacency matrix done!
2024-06-16 20:40:49.719 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:84 - Creating adjacency matrix based on provided graph...


Some services cannot be reached from some nodes of the graph. The nodes were removed from analysis


2024-06-16 20:41:01.055 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances ...
2024-06-16 20:43:27.383 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:93 - Adjacency matrix done!
2024-06-16 20:43:27.479 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:84 - Creating adjacency matrix based on provided graph...


...

2024-06-16 20:46:44.200 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances ...
2024-06-16 20:47:12.393 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:93 - Adjacency matrix done!
2024-06-16 20:47:12.432 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:84 - Creating adjacency matrix based on provided graph...


Some services cannot be reached from some nodes of the graph. The nodes were removed from analysis


2024-06-16 20:48:03.134 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances ...
2024-06-16 20:48:27.811 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:93 - Adjacency matrix done!


Some services cannot be reached from some nodes of the graph. The nodes were removed from analysis


/Users/sashamorozov/Documents/Code/NCCR/transport_frames/transport_frames/src/metrics/indicators.py:296: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  result = polygons_gdf.merge(length_sums, how='left', left_on='index', right_on='index').fillna(0)
/Users/sashamorozov/Documents/Code/NCCR/transport_frames/transport_frames/src/metrics/indicators.py:402: UserWarning: Approach is not set. Defaulting to 'primal'.
  ei = momepy.nx_to_gdf(inter)[1]


In [100]:
def read_json_file(filename):
    with open(filename, 'r') as f:
        json_object = json.load(f)

    result_dict = {}

    for key in json_object.keys():
        if isinstance(json_object[key], str):  # check if the value is a string
            try:
                gdf = gpd.read_file(json_object[key])
                result_dict[key] = gdf
            except:
                try:
                    result_dict[key] = json.loads(json_object[key])
                except json.JSONDecodeError:
                    result_dict[key] = json_object[key]
        else:
            result_dict[key] = json_object[key]

    return result_dict

d1 = read_json_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/indicators/Тульская_область/Тульская_область_indicators_gpsp.json')

print(d1.keys()) # check all possible keys
# a['connectivity'] # check what's stored inside

dict_keys(['aggregated_density', 'road_length_gdf', 'connectivity', 'connectivity_public_transport', 'to_fed_roads', 'to_region_admin_center', 'azs_availability', 'number_of_fuel_stations', 'train_stops_availability', 'number_of_train_stops', 'local_aero_availability', 'number_of_local_aero', 'number_of_bus_stops', 'train_paths_length', 'number_of_bus_routes'])


In [101]:
d1.keys()

dict_keys(['aggregated_density', 'road_length_gdf', 'connectivity', 'connectivity_public_transport', 'to_fed_roads', 'to_region_admin_center', 'azs_availability', 'number_of_fuel_stations', 'train_stops_availability', 'number_of_train_stops', 'local_aero_availability', 'number_of_local_aero', 'number_of_bus_stops', 'train_paths_length', 'number_of_bus_routes'])

In [111]:
gdf = d1['train_stops_availability']

In [3]:
d1['train_stops_availability'].explore()

NameError: name 'd1' is not defined

In [57]:
gdf['to_service'] = gdf['to_service'].astype(float)

In [60]:
vmin = 0.0
vmax = 28.0

In [113]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

# Ваши данные GeoDataFrame
gdf = d1['train_stops_availability']  # замените 'your_file.geojson' на ваш файл

# Проверка данных
print(gdf['to_service'].describe())
print(gdf['to_service'].isna().sum())

# Настройка colormap
cmap = plt.get_cmap('RdYlGn_r')

# Определение vmin и vmax
# vmin = gdf['density'].min()
vmax = gdf['to_service'].max()

vmin = 0.0
# vmax = 28.0

# Раскрасьте полигоны
gdf.explore(
    column='to_service', 
    cmap='RdYlGn_r', 
    legend=True, 
    figsize=(10, 8), 
    vmin=vmin, 
    vmax=vmax,
    tiles='CartoDB positron'
)


count    84.000000
mean     13.995229
std       9.382692
min       0.602600
25%       6.911192
50%      12.840179
75%      18.422983
max      37.364150
Name: to_service, dtype: float64
0


In [43]:
d1['connectivity_public_transport']

,index,name,geometry,to_service
0,0,Балашиха,"POLYGON ((37.77607 55.87273, 37.77666 55.86553...",14.977083
1,1,Богородский,"POLYGON ((38.05194 55.77862, 38.05258 55.77690...",16.295167
2,2,Бронницы,"POLYGON ((38.19609 55.43117, 38.19627 55.42981...",17.785958
3,3,Власиха,"POLYGON ((37.16482 55.67983, 37.16516 55.67945...",21.772333
4,4,Волоколамский,"POLYGON ((35.60228 55.73550, 35.60684 55.73139...",21.516125
5,5,Воскресенск,"POLYGON ((38.35243 55.44720, 38.35655 55.44747...",19.604500
6,6,Восход,"POLYGON ((36.43647 55.94057, 36.44311 55.93299...",15.981917
7,7,Дзержинский,"POLYGON ((37.79356 55.62296, 37.79643 55.62121...",16.032333
8,8,Дмитровский,"POLYGON ((36.97760 56.46231, 36.98151 56.45975...",17.387125
9,9,Долгопрудный,"POLYGON ((37.45482 55.97277, 37.45496 55.97262...",17.466583


In [45]:
ind_dict1 = {

    'to_fed_roads': {
        'column': 'to_service',
        '№ п/п':'3.1.1.2',
        'Название хранимое':'Средняя удаленность до федеральных транспортных магистралей',
        'ед.изм.': 'км'

    },

    'to_region_admin_center': {
        'column': 'to_service',
        '№ п/п':'3.1.1.4',
        'Название хранимое':'Средняя удаленность до центра региона',
        'ед.изм.': 'км'

    },

    'connectivity': {
        'column': 'to_service',
        '№ п/п':'3.1.2',
        'Название хранимое':'Связанность',
        'ед.изм.': 'час'

    },

    # 'connectivity_public_transport': {
    #     'column': 'to_service',
    #     '№ п/п':'3.1.2',
    #     'Название хранимое':'Связанность на ОТ',
    #     'ед.изм.': 'час'

    # },

    'aggregated_density': {
        'column': 'density',
        '№ п/п':'3.1.3',
        'Название хранимое':'Плотность улично-дорожной сети',
        'ед.изм.': 'км/км2'

    },

    'road_length_gdf': {
        'reg1_length': {
            'column': 'reg1_length',
            '№ п/п':'3.1.4.1',
            'Название хранимое':'Протяженность дорог федерального значения',
            'ед.изм.': 'км'
        },
        'reg2_length': {
            'column': 'reg2_length',
            '№ п/п':'3.1.4.2',
            'Название хранимое':'Протяженность дорог регионального значения',
            'ед.изм.': 'км'
        },
        'reg3_length': {
            'column': 'reg3_length',
            '№ п/п':'3.1.4.3',
            'Название хранимое':'Протяженность дорог местного значения',
            'ед.изм.': 'км'
        }
    }, 

    'number_of_bus_routes': {
        'column': 'number_of_routes',
        '№ п/п':'3.1.5.1.1',
        'Название хранимое':'Количество маршрутов общественного транспорта',
        'ед.изм.': 'ед'

    },

    'number_of_bus_stops': {
        'column': 'service_count',
        '№ п/п':'3.1.5.1.2',
        'Название хранимое':'Количество остановок общественного транспорта',
        'ед.изм.': 'ед'

    },
    
    'number_of_fuel_stations': {
        'column': 'service_count',
        '№ п/п':'3.1.5.2.1',
        'Название хранимое':'Количество автозаправочных станций',
        'ед.изм.': 'ед'

    },

    'azs_availability': {
        'column': 'to_service',
        '№ п/п':'3.1.5.2.2',
        'Название хранимое':'Средняя доступность автозаправочных станций',
        'ед.изм.': 'мин'

    },

     'train_paths_length': {
        'column': 'total_length_km',
        '№ п/п':'3.2.1',
        'Название хранимое':'Общая протяженность железнодорожных путей',
        'ед.изм.': 'км'

    },

    'number_of_train_stops': {
        'column': 'service_count',
        '№ п/п':'3.2.2',
        'Название хранимое':'Количество остановок железнодорожного транспорта',
        'ед.изм.': 'ед'

    },

    'train_stops_availability': {
        'column': 'to_service',
        '№ п/п':'3.2.3',
        'Название хранимое':'Средняя доступность остановок железнодорожного транспорта',
        'ед.изм.': 'мин'

    },

    'number_of_international_aero': {
        'column': 'service_count',
        '№ п/п':'3.3.1.1',
        'Название хранимое':'Количество международных аэропортов',
        'ед.изм.': 'ед'

    },

    'number_of_local_aero': {
        'column': 'service_count',
        '№ п/п':'3.3.1.2',
        'Название хранимое':'Количество аэропортов местного значения',
        'ед.изм.': 'ед'

    },

    'international_aero_availability': {
        'column': 'to_service',
        '№ п/п':'3.3.2.1',
        'Название хранимое':'Средняя доступность международных аэропортов',
        'ед.изм.': 'мин'

    },

    'local_aero_availability': {
        'column': 'to_service',
        '№ п/п':'3.3.2.2',
        'Название хранимое':'Средняя доступность аэропортов местного значения',
        'ед.изм.': 'мин'

    },

    'number_of_ports': {
        'column': 'service_count',
        '№ п/п':'3.4.1',
        'Название хранимое':'Количество портов',
        'ед.изм.': 'ед'

   }

}

gdfs = []

for gdf_name, gdf_dict in ind_dict1.items():
    tmp_gdf = d1[gdf_name].copy()
    if 'column' in gdf_dict:
        # Обработка верхнеуровневых словарей
        tmp_gdf = tmp_gdf[['name', gdf_dict['column']]].rename(columns={
            'name': 'Территория',
            gdf_dict['column']: 'Значение'
        })
        for key, value in gdf_dict.items():
            if key == 'column':
                continue
            tmp_gdf[key] = value
        gdfs.append(tmp_gdf)
    else:
        # Обработка вложенных словарей
        for sub_key, sub_dict in gdf_dict.items():
            tmp_gdf_sub = tmp_gdf[['name', sub_dict['column']]].rename(columns={
                'name': 'Территория',
                sub_dict['column']: 'Значение'
            })
            for key, value in sub_dict.items():
                if key == 'column':
                    continue
                tmp_gdf_sub[key] = value
            gdfs.append(tmp_gdf_sub)

# Объединяем все в один DataFrame
dfdf1 = pd.concat(gdfs)
dfdf1 = dfdf1[['№ п/п','Название хранимое','ед.изм.','Значение','Территория']]

dfdf1 = dfdf1.assign(
    Источник='modeled',
    Период='2024'
)
# dfdf1.to_excel('Московская_область.xlsx', index=False)
dfdf1


,№ п/п,Название хранимое,ед.изм.,Значение,Территория,Источник,Период
0,3.1.1.2,Средняя удаленность до федеральных транспортны...,км,8.602783,Балашиха,modeled,2024
1,3.1.1.2,Средняя удаленность до федеральных транспортны...,км,4.392374,Богородский,modeled,2024
2,3.1.1.2,Средняя удаленность до федеральных транспортны...,км,1.108131,Бронницы,modeled,2024
3,3.1.1.2,Средняя удаленность до федеральных транспортны...,км,1.754023,Власиха,modeled,2024
4,3.1.1.2,Средняя удаленность до федеральных транспортны...,км,17.982971,Волоколамский,modeled,2024
...,...,...,...,...,...,...,...
55,3.4.1,Количество портов,ед,0.000000,Шаховская,modeled,2024
56,3.4.1,Количество портов,ед,0.000000,Щелково,modeled,2024
57,3.4.1,Количество портов,ед,0.000000,Павлово-Посадский,modeled,2024
58,3.4.1,Количество портов,ед,0.000000,Электросталь,modeled,2024


# Settlement

In [22]:
d2 = indicators.indicator_area(citygraph, polygons188, points, inter,region_capital, 
                               fuel, stops, international_aero, aero, ferry,train_paths,bus_stops, crs=local_crs)

for k,v in d2.items():
    if isinstance(v,gpd.GeoDataFrame):
        cols_to_drop = [col for col in v.columns if col not in ['index', 'name', 'layer', 'density', 'geometry', 'reg1_length', 'reg2_length', 'reg3_length', 'to_service', 'index', 'number_of_routes', 'service_count', 'total_length_km']]
        v.drop(cols_to_drop, axis=1, inplace=True)
create_json_file(d2,f"../preprocessed/{region_name}/{region_name}_indicators_gpsp.json")

2024-06-16 04:20:50.064 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:84 - Creating adjacency matrix based on provided graph...
2024-06-16 04:21:02.576 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances ...
2024-06-16 04:23:58.977 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:93 - Adjacency matrix done!
/Users/sashamorozov/Documents/Code/NCCR/transport_frames/.venv/lib/python3.10/site-packages/numpy/core/_methods.py:118: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
2024-06-16 04:23:59.242 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:84 - Creating adjacency matrix based on provided graph...
2024-06-16 04:25:31.614 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances ...
2024-06-16 04:29:53.695 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:93 - Adjacency matrix done!
/Users/sashamorozov/Documents

Some services cannot be reached from some nodes of the graph. The nodes were removed from analysis


2024-06-16 04:32:46.985 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances ...
2024-06-16 04:32:48.898 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:93 - Adjacency matrix done!
2024-06-16 04:32:48.925 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:84 - Creating adjacency matrix based on provided graph...


Some services cannot be reached from some nodes of the graph. The nodes were removed from analysis


2024-06-16 04:33:21.485 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances ...
2024-06-16 04:33:37.286 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:93 - Adjacency matrix done!
2024-06-16 04:33:37.324 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:84 - Creating adjacency matrix based on provided graph...


...

Some services cannot be reached from some nodes of the graph. The nodes were removed from analysis


2024-06-16 04:34:45.215 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances ...
2024-06-16 04:34:51.978 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:93 - Adjacency matrix done!
2024-06-16 04:34:52.011 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:84 - Creating adjacency matrix based on provided graph...


Some services cannot be reached from some nodes of the graph. The nodes were removed from analysis


2024-06-16 04:34:57.280 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances ...
2024-06-16 04:35:01.085 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:93 - Adjacency matrix done!
/Users/sashamorozov/Documents/Code/NCCR/transport_frames/transport_frames/src/metrics/indicators.py:402: UserWarning: Approach is not set. Defaulting to 'primal'.
  ei = momepy.nx_to_gdf(inter)[1]


Some services cannot be reached from some nodes of the graph. The nodes were removed from analysis


In [108]:
def read_json_file(filename):
    with open(filename, 'r') as f:
        json_object = json.load(f)

    result_dict = {}

    for key in json_object.keys():
        if isinstance(json_object[key], str):  # check if the value is a string
            try:
                gdf = gpd.read_file(json_object[key])
                result_dict[key] = gdf
            except:
                try:
                    result_dict[key] = json.loads(json_object[key])
                except json.JSONDecodeError:
                    result_dict[key] = json_object[key]
        else:
            result_dict[key] = json_object[key]

    return result_dict

d2 = read_json_file('/Users/sashamorozov/Downloads/Архив/ЛО/Ленинградская_область/Ленинградская_область_indicators_mr.json')

print(d2.keys()) # check all possible keys
# a['connectivity'] # check what's stored inside

dict_keys(['aggregated_density', 'road_length_gdf', 'connectivity', 'connectivity_public_transport', 'to_fed_roads', 'to_region_admin_center', 'azs_availability', 'train_stops_availability', 'international_aero_availability', 'local_aero_availability', 'port_availability', 'number_of_bus_routes', 'number_of_bus_stops', 'number_of_fuel_stations', 'number_of_train_stops', 'number_of_international_aero', 'number_of_local_aero', 'number_of_ports', 'train_paths_length'])


In [23]:
d2.keys()

dict_keys(['aggregated_density', 'road_length_gdf', 'connectivity', 'connectivity_public_transport', 'to_fed_roads', 'to_region_admin_center', 'azs_availability', 'number_of_fuel_stations', 'train_stops_availability', 'number_of_train_stops', 'international_aero_availability', 'number_of_international_aero', 'local_aero_availability', 'number_of_local_aero', 'port_availability', 'number_of_ports', 'number_of_bus_stops', 'train_paths_length', 'number_of_bus_routes'])

In [25]:
d2['connectivity_public_transport']

,index,name,layer,geometry,to_service
0,0,Самойловское сельское поселение,Бокситогорский муниципальный район,"MULTIPOLYGON (((34.42169 59.68391, 34.42428 59...",6.774613
1,1,Большедворское сельское поселение,Бокситогорский муниципальный район,"MULTIPOLYGON (((34.30682 59.71494, 34.30791 59...",6.878877
2,2,Пикалевское городское поселение,Бокситогорский муниципальный район,"MULTIPOLYGON (((34.14251 59.48759, 34.13616 59...",6.106106
3,3,Борское сельское поселение,Бокситогорский муниципальный район,"MULTIPOLYGON (((34.05997 59.15087, 34.06007 59...",6.456924
4,4,Бокситогорское городское поселение,Бокситогорский муниципальный район,"MULTIPOLYGON (((34.06836 59.40315, 34.05535 59...",6.051439
...,...,...,...,...,...
183,183,Фёдоровское городское поселение,Тосненский муниципальный район,"MULTIPOLYGON (((30.52049 59.69087, 30.52316 59...",4.120390
184,184,Нурминское сельское поселение,Тосненский муниципальный район,"MULTIPOLYGON (((31.01271 59.63362, 31.01394 59...",4.664563
185,185,Красноборское городское поселение,Тосненский муниципальный район,"MULTIPOLYGON (((30.58336 59.64993, 30.58367 59...",3.861638
186,186,Трубникоборское сельское поселение,Тосненский муниципальный район,"MULTIPOLYGON (((31.13126 59.08942, 31.13522 59...",5.736096


In [26]:
ind_dict2 = {

    'to_fed_roads': {
        'column': 'to_service',
        '№ п/п':'3.1.1.2',
        'Название хранимое':'Средняя удаленность до федеральных транспортных магистралей',
        'ед.изм.': 'км'

    },

    'to_region_admin_center': {
        'column': 'to_service',
        '№ п/п':'3.1.1.4',
        'Название хранимое':'Средняя удаленность до центра региона',
        'ед.изм.': 'км'

    },

    'connectivity': {
        'column': 'to_service',
        '№ п/п':'3.1.2',
        'Название хранимое':'Связанность',
        'ед.изм.': 'час'

    },

    'connectivity_public_transport': {
        'column': 'to_service',
        '№ п/п':'3.1.2',
        'Название хранимое':'Связанность на ОТ',
        'ед.изм.': 'час'

    },

    'aggregated_density': {
        'column': 'density',
        '№ п/п':'3.1.3',
        'Название хранимое':'Плотность улично-дорожной сети',
        'ед.изм.': 'км/км2'

    },

    'road_length_gdf': {
        'reg1_length': {
            'column': 'reg1_length',
            '№ п/п':'3.1.4.1',
            'Название хранимое':'Протяженность дорог федерального значения',
            'ед.изм.': 'км'
        },
        'reg2_length': {
            'column': 'reg2_length',
            '№ п/п':'3.1.4.2',
            'Название хранимое':'Протяженность дорог регионального значения',
            'ед.изм.': 'км'
        },
        'reg3_length': {
            'column': 'reg3_length',
            '№ п/п':'3.1.4.3',
            'Название хранимое':'Протяженность дорог местного значения',
            'ед.изм.': 'км'
        }
    }, 

    'number_of_bus_routes': {
        'column': 'number_of_routes',
        '№ п/п':'3.1.5.1.1',
        'Название хранимое':'Количество маршрутов общественного транспорта',
        'ед.изм.': 'ед'

    },

    'number_of_bus_stops': {
        'column': 'service_count',
        '№ п/п':'3.1.5.1.2',
        'Название хранимое':'Количество остановок общественного транспорта',
        'ед.изм.': 'ед'

    },
    
    'number_of_fuel_stations': {
        'column': 'service_count',
        '№ п/п':'3.1.5.2.1',
        'Название хранимое':'Количество автозаправочных станций',
        'ед.изм.': 'ед'

    },

    'azs_availability': {
        'column': 'to_service',
        '№ п/п':'3.1.5.2.2',
        'Название хранимое':'Средняя доступность автозаправочных станций',
        'ед.изм.': 'мин'

    },

     'train_paths_length': {
        'column': 'total_length_km',
        '№ п/п':'3.2.1',
        'Название хранимое':'Общая протяженность железнодорожных путей',
        'ед.изм.': 'км'

    },

    'number_of_train_stops': {
        'column': 'service_count',
        '№ п/п':'3.2.2',
        'Название хранимое':'Количество остановок железнодорожного транспорта',
        'ед.изм.': 'ед'

    },

    'train_stops_availability': {
        'column': 'to_service',
        '№ п/п':'3.2.3',
        'Название хранимое':'Средняя доступность остановок железнодорожного транспорта',
        'ед.изм.': 'мин'

    },

    'number_of_international_aero': {
        'column': 'service_count',
        '№ п/п':'3.3.1.1',
        'Название хранимое':'Количество международных аэропортов',
        'ед.изм.': 'ед'

    },

    'number_of_local_aero': {
        'column': 'service_count',
        '№ п/п':'3.3.1.2',
        'Название хранимое':'Количество аэропортов местного значения',
        'ед.изм.': 'ед'

    },

    'international_aero_availability': {
        'column': 'to_service',
        '№ п/п':'3.3.2.1',
        'Название хранимое':'Средняя доступность международных аэропортов',
        'ед.изм.': 'мин'

    },

    'local_aero_availability': {
        'column': 'to_service',
        '№ п/п':'3.3.2.2',
        'Название хранимое':'Средняя доступность аэропортов местного значения',
        'ед.изм.': 'мин'

    },

    'number_of_ports': {
        'column': 'service_count',
        '№ п/п':'3.4.1',
        'Название хранимое':'Количество портов',
        'ед.изм.': 'ед'

   }

}

gdfs = []

for gdf_name, gdf_dict in ind_dict2.items():
    tmp_gdf = d2[gdf_name].copy()
    if 'column' in gdf_dict:
        # Обработка верхнеуровневых словарей
        tmp_gdf = tmp_gdf[['name', gdf_dict['column']]].rename(columns={
            'name': 'Территория',
            gdf_dict['column']: 'Значение'
        })
        for key, value in gdf_dict.items():
            if key == 'column':
                continue
            tmp_gdf[key] = value
        gdfs.append(tmp_gdf)
    else:
        # Обработка вложенных словарей
        for sub_key, sub_dict in gdf_dict.items():
            tmp_gdf_sub = tmp_gdf[['name', sub_dict['column']]].rename(columns={
                'name': 'Территория',
                sub_dict['column']: 'Значение'
            })
            for key, value in sub_dict.items():
                if key == 'column':
                    continue
                tmp_gdf_sub[key] = value
            gdfs.append(tmp_gdf_sub)

# Объединяем все в один DataFrame
dfdf2 = pd.concat(gdfs)
dfdf2 = dfdf2[['№ п/п','Название хранимое','ед.изм.','Значение','Территория']]

dfdf2 = dfdf2.assign(
    Источник='modeled',
    Период='2024'
)
# dfdf.to_excel('Тульская_область_район.xlsx', index=False)
dfdf2


,№ п/п,Название хранимое,ед.изм.,Значение,Территория,Источник,Период
0,3.1.1.2,Средняя удаленность до федеральных транспортны...,км,17.014939,Самойловское сельское поселение,modeled,2024
1,3.1.1.2,Средняя удаленность до федеральных транспортны...,км,15.386207,Большедворское сельское поселение,modeled,2024
2,3.1.1.2,Средняя удаленность до федеральных транспортны...,км,1.016858,Пикалевское городское поселение,modeled,2024
3,3.1.1.2,Средняя удаленность до федеральных транспортны...,км,27.648422,Борское сельское поселение,modeled,2024
4,3.1.1.2,Средняя удаленность до федеральных транспортны...,км,7.247291,Бокситогорское городское поселение,modeled,2024
...,...,...,...,...,...,...,...
183,3.4.1,Количество портов,ед,0.000000,Фёдоровское городское поселение,modeled,2024
184,3.4.1,Количество портов,ед,0.000000,Нурминское сельское поселение,modeled,2024
185,3.4.1,Количество портов,ед,0.000000,Красноборское городское поселение,modeled,2024
186,3.4.1,Количество портов,ед,0.000000,Трубникоборское сельское поселение,modeled,2024


In [46]:
dfdf3 = pd.concat([dfdf,dfdf1],ignore_index=True)
dfdf3

,№ п/п,Название хранимое,ед.изм.,Значение,Территория,Источник,Период
0,3.1.1.2,Средняя удаленность до федеральных транспортны...,км,9.919423,Московская область,modeled,2024
1,3.1.1.4,Средняя удаленность до центра региона,км,97.182432,Московская область,modeled,2024
2,3.1.3,Плотность улично-дорожной сети,км/км2,2645.678000,Московская область,modeled,2024
3,3.1.4.1,Протяженность дорог федерального значения,км,5720.435614,Московская область,modeled,2024
4,3.1.4.2,Протяженность дорог регионального значения,км,33730.126723,Московская область,modeled,2024
...,...,...,...,...,...,...,...
1153,3.4.1,Количество портов,ед,0.000000,Шаховская,modeled,2024
1154,3.4.1,Количество портов,ед,0.000000,Щелково,modeled,2024
1155,3.4.1,Количество портов,ед,0.000000,Павлово-Посадский,modeled,2024
1156,3.4.1,Количество портов,ед,0.000000,Электросталь,modeled,2024


In [47]:
dfdf3.to_excel('Московская_область.xlsx', index=False)

# Area

In [22]:
d3 = indicators.indicator_territory(citygraph, neud,
                                    regions_gdf=polygons18,
                                    districts_gdf=polygons188,
                                    region_capital=region_capital,
                                    region_centers=points18,
                                    district_centers=polygons188,
                                    settlement_centers=points,
                                    fuel=fuel,
                                    train_stops=stops,
                                    international_aero=international_aero,
                                    aero=aero,
                                    ports=ferry,
                                    water_objects=water_objects, 
                                    oopt=oopt, 
                                    train_paths=train_paths, 
                                    inter = None,
                                    bus_stops=bus_stops, 
                                    bus_routes=bus_routes,crs=local_crs)

2024-06-17 13:44:27.124 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:84 - Creating adjacency matrix based on provided graph...
2024-06-17 13:44:41.351 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances ...
2024-06-17 13:44:41.903 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:93 - Adjacency matrix done!
2024-06-17 13:44:41.909 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:84 - Creating adjacency matrix based on provided graph...
2024-06-17 13:44:48.283 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances ...
2024-06-17 13:44:48.667 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:93 - Adjacency matrix done!
2024-06-17 13:44:48.694 | INFO     | dongraphio.dongraphio:get_adjacency_matrix:84 - Creating adjacency matrix based on provided graph...
2024-06-17 13:45:04.474 | DEBUG    | dongraphio.base_models.matrix_logic:get_adjacency_matrix:52 - Calculating distances

In [27]:
d3.keys()

dict_keys(['to_fed_roads', 'to_region_admin_center', 'connectivity_region_center', 'connectivity_district_center', 'connectivity_settlement', 'density', 'number_of_bus_routes', 'number_of_bus_stops', 'train_paths_length', 'number_of_fuel_stations', 'azs_availability', 'number_of_international_aero', 'international_aero_availability', 'number_of_local_aero', 'local_aero_availability', 'number_of_train_stops', 'train_stops_availability', 'number_of_ports', 'ports_availability', 'oopt_availability', 'number_of_water_objects', 'water_objects_availability'])

In [1]:
d3

NameError: name 'd3' is not defined

In [26]:
for k,v in d3.items():
    if isinstance(v,gpd.GeoDataFrame):
        cols_to_drop = [col for col in v.columns if col not in ['index', 'name', 'layer', 'density', 'geometry', 'reg1_length', 'reg2_length', 'reg3_length', 'to_service', 'index', 'number_of_routes', 'service_count', 'total_length_km']]
        v.drop(cols_to_drop, axis=1, inplace=True)
create_json_file(d3,f"../preprocessed/{region_name}/{region_name}_indicators_Сиверск.json")

# end